In [36]:
import os
from dotenv import load_dotenv

# 1. Load the hidden environment variables
load_dotenv()  

# 2. Fetch the key (make sure there is no space between GROQ and _API_KEY)
api_key = os.getenv("GROQ_API_KEY")

# 3. Quick test to confirm it loaded successfully
if api_key:
    print("Success: API Key loaded!")
    print(f"Key starts with: {api_key[:8]}...") # Shows 'gsk_xxxx' without revealing the whole key
else:
    print("Error: Could not find GROQ_API_KEY. Check your .env file spelling!")

Success: API Key loaded!
Key starts with: gsk_MRMG...


In [39]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [40]:
import os 
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
llm_model = ChatGroq(
    model_name= "llama-3.3-70b-versatile",
    temperature= 0.6,
    max_retries=2
)

In [48]:
prompt_template = PromptTemplate(
    input_variables= ['cuisine'],
    template= "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this, only one name, no explination."
)
prompt_template.template.format(cuisine="mexican")

'I want to open a restaurant for mexican food. Suggest a fancy name for this, only one name, no explination.'

In [ ]:
llm_restaurant_name_chain = prompt_template | llm_model | StrOutputParser()

llm_response_1 = llm_restaurant_name_chain.invoke({"cuisine" : "mexican"})
print(llm_response_1)

"El Fuego de Oro"


In [ ]:
# Define LLM
llm_model = llm_model

# Step 1: Restaurant name chain
prompt_template = PromptTemplate(
    input_variables= ['cuisine'],
    template= "I want to open a restaurant for {cuisine} food. Suggest a fancy name for this. Return ONLY one name, no explanation."
)

llm_restaurant_name_chain = prompt_template | llm_model | StrOutputParser() 

# Step 2: Menu chain
prompt_template = PromptTemplate(
    input_variables= ['menu_card'],
    template= """Suggest menu items for {menu_card}. Return ONLY a comma-separated list, no explanations, no translations."""
)

llm_menu_chain = prompt_template | llm_model | StrOutputParser()
 

In [ ]:
# overall_chain = llm_restaurant_name_chain | llm_menu_chain | StrOutputParser()

# llm_response_2 = overall_chain.invoke({"cuisine":"german"})
# print(llm_response_2)

Schweinshaxe, Sauerbraten, Schnitzel, Spätzle, Bratwurst, Currywurst, Leberkäse, Weisswurst, Sauerkraut, Brezen, Apple Strudel, Black Forest Cake, Berliner Pfannkuchen


In [ ]:
# # Full sequential pipeline (replaces SimpleSequentialChain)
# full_chain = llm_restaurant_name_chain | (lambda name: {"menu_card": name}) | llm_menu_chain

# result = full_chain.invoke({"cuisine": "Italian"})
# print(result)

Bruschetta, Insalata Caprese, Fettuccine Alfredo, Pollo alla Cacciatora, Risotto con Funghi, Spaghetti Bolognese, Lasagna Classica, Cannoli, Tiramisù, Gelato di Cioccolato


In [105]:
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    {"cuisine" : llm_restaurant_name_chain} 
    | RunnablePassthrough.assign(
        menu_card = lambda x: llm_menu_chain.invoke({"cuisine": x["cuisine"]})
    )
)

result = full_chain.invoke({"cuisine": "Italian"})
print(result)

KeyError: "Input to PromptTemplate is missing variables {'menu_card'}.  Expected: ['menu_card'] Received: ['cuisine']\nNote: if you intended {menu_card} to be part of the string and not a variable, please escape it with double curly braces like: '{{menu_card}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "